In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Membaca data dari HDFS
df_transaksi_tugas = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True, inferSchema=True
)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True, inferSchema=True
)

print(f"Data Berhasil Memuat dari HDFS:")
print(f"- Transaksi : {df_transaksi_tugas.count()} baris")
print(f"- Cabang    : {df_cabang.count()} baris")
print(f"- Menu      : {df_menu.count()} baris")

26/10/11 13:00:56 WARN Utils: Your hostname, raam resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/11 13:00:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 13:01:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/11 13:01:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

Data Berhasil Memuat dari HDFS:


- Transaksi : 4000 baris
- Cabang    : 3 baris
- Menu      : 15 baris


In [2]:
# Penggabungan tabel transaksi, cabang, dan menu
df_gabungan_tugas = df_transaksi_tugas \
    .join(df_cabang, on="cabang_id", how="inner") \
    .join(df_menu, on="menu_id", how="inner")

# Menambahkan kolom total_penjualan = qty * harga
df_gabungan_tugas = df_gabungan_tugas.withColumn(
    "total_penjualan", col("qty") * col("harga")
)

df_gabungan_tugas.select(
    "trx_id", "nama_cabang", "nama_menu", "kategori_menu", "qty", "harga", "total_penjualan"
).show(5)

+------+--------------------+---------+--------------+---+-----+---------------+
|trx_id|         nama_cabang|nama_menu| kategori_menu|qty|harga|total_penjualan|
+------+--------------------+---------+--------------+---+-----+---------------+
|   KN0|Kopi Nusantara Ma...|  Menu-14|          Kopi|  4|18000|          72000|
|   KN1|Kopi Nusantara Ma...|  Menu-11|          Kopi|  4|22000|          88000|
|   KN2|Kopi Nusantara Ma...|  Menu-11|          Kopi|  3|22000|          66000|
|   KN3|Kopi Nusantara Se...|  Menu-11|          Kopi|  1|22000|          22000|
|   KN4|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|  4|35000|         140000|
+------+--------------------+---------+--------------+---+-----+---------------+
only showing top 5 rows



In [3]:
# Ringkasan total penjualan per kombinasi cabang dan menu
ringkasan_menu_cabang = df_gabungan_tugas.groupBy("nama_cabang", "nama_menu", "kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Window function terpartisi per nama_cabang
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

# Memfilter top-2 menu per cabang
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                 .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show()

[Stage 24:>                                                         (0 + 1) / 1]

+--------------------+---------+--------------+---------------+---------+
|         nama_cabang|nama_menu| kategori_menu|total_penjualan|peringkat|
+--------------------+---------+--------------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|Makanan Ringan|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|Makanan Ringan|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|Makanan Ringan|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|Makanan Ringan|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|      Non-Kopi|        8160000|        2|
+--------------------+---------+--------------+---------------+---------+



In [5]:
# Membuat TempView
df_gabungan_tugas.createOrReplaceTempView("transaksi_kopi")

# Kueri SQL Murni
kontribusi_barista = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM transaksi_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

kontribusi_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [6]:
# Membuat direktori processed
!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/processed

# Menyimpan data terpartisi berdasarkan kategori_menu
df_gabungan_tugas.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

# Verifikasi dengan membaca ulang dari HDFS
df_terverifikasi = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")

print(f"Verifikasi Berhasil! Total baris terhitung: {df_terverifikasi.count()}")
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan

[Stage 42:>                                                         (0 + 1) / 1]

Verifikasi Berhasil! Total baris terhitung: 4000


Found 4 items
-rw-r--r--   3 raamxh supergroup          0 2026-10-11 13:03 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - raamxh supergroup          0 2026-10-11 13:03 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - raamxh supergroup          0 2026-10-11 13:03 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - raamxh supergroup          0 2026-10-11 13:03 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi


In [7]:
ringkasan_eksekutif_tugas = df_gabungan_tugas.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

ringkasan_eksekutif_tugas.show(truncate=False)

[Stage 47:>                                                         (0 + 1) / 1]

+-------------------------+----------+--------------+------------------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+------------------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000                |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000                |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000                |1305            |64862.06896551724        |
+-------------------------+----------+--------------+------------------------+----------------+-------------------------+



Berdasarkan hasil pengolahan data triwulan jaringan kedai Kopi Nusantara, cabang Kopi Nusantara Semarang mencatatkan kinerja performa tertinggi dengan total penjualan sebesar Rp90.542.000 dan volume transaksi sebanyak 1.393 transaksi. Sebaliknya, cabang Kopi Nusantara Yogyakarta menjadi cabang dengan performa terlemah, memperoleh pendapatan terendah yaitu Rp84.645.000 serta rata-rata nilai transaksi (average order value) terkecil sebesar Rp64.862,07 per transaksi.   